In [0]:
from pyspark.sql import functions as F

bronze_order_reviews = spark.table("brazilian_ecommerce.prod.bronze_orders_review")

def clean_null_string(col: str):
    return F.when(F.lower(F.trim(F.col(col))) == "null", F.lit(None)).otherwise(F.col(col))

staging_order_reviews = (
    bronze_order_reviews
    .withColumn("review_comment_title", clean_null_string("review_comment_title"))
    .withColumn("review_comment_message", clean_null_string("review_comment_message"))

    .withColumn("review_score", F.col("review_score").cast("int"))
    .filter(F.col("review_score").between(1, 5))   # enforce valid score range, drop anything outside it

    .withColumn("review_comment_title", F.trim(F.lower(F.col("review_comment_title"))))
    .withColumn("review_comment_message", F.trim(F.lower(F.col("review_comment_message"))))

    .withColumn("review_creation_date", F.to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))

    .withColumn(
        "days_creation_to_answer",
        F.datediff(F.to_date("review_answer_timestamp"), F.to_date("review_creation_date"))
    )

    .filter(F.col("review_id").isNotNull())
    .dropDuplicates(["review_id"])
    .withColumn("_loaded_at", F.current_timestamp())
)

full_table_name = "brazilian_ecommerce.prod.staging_order_reviews"

(
    staging_order_reviews.write
    .mode("overwrite")
    .option("comment", "order reviews staging table")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

# primary key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN review_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_staging_order_reviews PRIMARY KEY (review_id)
""")

# foreign key
spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT fk_staging_order_reviews_order_id
    FOREIGN KEY (order_id)
    REFERENCES brazilian_ecommerce.prod.staging_orders (order_id)
""")

print(f"Loaded {staging_order_reviews.count()} rows into {full_table_name}")